# Pipeline Atribut SI & Matriks OSRM

Rangkuman Notebook 4-7 menjadi satu pipeline end-to-end untuk menyiapkan data backend:

1. **Klasifikasi jalan (C2 -> `road_class`)** — Overpass OSM, skala ordinal 1-5.
2. **Jarak ke faskes (C3 -> `dist_faskes_m`)** — OSRM Route, jaringan jalan aktual.
3. **Merge -> `floods_final.csv`** — skema yang dibaca `backend/app/main.py`.
4. **Matriks jarak & waktu -> `.npy`** — OSRM Table antar seluruh node model.

Dijalankan sekali secara offline saat dataset berubah (skenario banjir statis).
Input: `floods-skenario1.csv`, `faskes.csv`, `depo.csv`, `if.csv` di `backend/app/data/`.

In [1]:
%pip install -q geopandas shapely requests pandas numpy tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import requests
from shapely.geometry import LineString
from tqdm import tqdm

# --- Path input/output ---
DATA_DIR = Path("../backend/app/data")
SCENARIO_FILE = DATA_DIR / "floods-skenario1.csv"   # skenario genangan terpilih
FASKES_FILE, DEPO_FILE, IF_FILE = DATA_DIR / "faskes.csv", DATA_DIR / "depo.csv", DATA_DIR / "if.csv"
OUT = Path("output_pipeline"); OUT.mkdir(exist_ok=True)

# --- Bounding box Surabaya (samakan dengan backend main.py) ---
BBOX = (-7.38, 112.58, -7.13, 112.87)               # south, west, north, east
SBY_LAT_MIN, SBY_LAT_MAX = -7.38, -7.13
SBY_LON_MIN, SBY_LON_MAX = 112.58, 112.87
CRS_WGS84, CRS_UTM49S = "EPSG:4326", "EPSG:32749"

# --- Klasifikasi jalan (C2): tag highway -> ordinal 1-5, nilai tertinggi = paling kritikal ---
HIGHWAY_ORDINAL = {
    "trunk": 5, "primary": 5, "secondary": 4, "tertiary": 3,
    "residential": 2, "service": 1, "living_street": 1,
}
SEARCH_RADIUS_M = 100                                # radius cari ruas di sekitar genangan

# --- OSRM (jarak faskes + matriks antar-node) ---
OSRM_BASE = "https://router.project-osrm.org"
K_CANDIDATES = 5                                     # faskes kandidat terdekat (Haversine) yg dicek via OSRM
BLOCK_SIZE = 40                                      # ukuran blok Table Service
REQUEST_DELAY_S, MAX_RETRIES, RETRY_WAIT_S = 1.0, 3, 5
EARTH_RADIUS_M = 6_371_000.0

OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
    "https://overpass.private.coffee/api/interpreter",
]
USER_AGENT = "MDCVRP-IF-SI-TA/1.0 (student research; Surabaya flood pumping)"

print("Setup selesai.")

Setup selesai.


In [3]:
def _bbox_filter(df):
    m = df["lat"].between(SBY_LAT_MIN, SBY_LAT_MAX) & df["lon"].between(SBY_LON_MIN, SBY_LON_MAX)
    return df[m].reset_index(drop=True)

# Skema backend lama memakai Latitude/Longitude; skenario baru sudah lat/lon.
LATLON_RENAME = {"Latitude": "lat", "Longitude": "lon", "latitude": "lat", "longitude": "lon"}

def read_points(path, rename=LATLON_RENAME):
    # Baca CSV, normalkan lat/lon, buang koordinat kosong, filter bbox Surabaya.
    df = pd.read_csv(path).rename(columns=rename)
    df["lat"] = pd.to_numeric(df["lat"], errors="coerce")
    df["lon"] = pd.to_numeric(df["lon"], errors="coerce")
    return _bbox_filter(df.dropna(subset=["lat", "lon"]).reset_index(drop=True))

# Muat sekali; urutan baris ini yang jadi id F_XXXX dan urutan node genangan di matriks.
floods = read_points(SCENARIO_FILE)
floods["id"] = [f"F_{i:04d}" for i in range(len(floods))]
print(f"Genangan: {len(floods)}")

Genangan: 34


## 1. Klasifikasi jalan (C2 -> `road_class`)

Tarik seluruh ruas jalan Surabaya dari OSM, petakan tag `highway` ke ordinal 1-5,
lalu untuk tiap genangan ambil ordinal tertinggi dalam `SEARCH_RADIUS_M` (fallback: ruas terdekat).

In [4]:
def fetch_overpass(query):
    headers = {"User-Agent": USER_AGENT, "Accept": "application/json"}
    for endpoint in OVERPASS_ENDPOINTS:                 # coba tiap mirror
        for _ in range(MAX_RETRIES):
            try:
                r = requests.post(endpoint, data={"data": query}, headers=headers, timeout=180)
                if r.status_code == 200:
                    return r.json()
            except requests.RequestException:
                pass
            time.sleep(RETRY_WAIT_S)
    raise RuntimeError("Semua Overpass endpoint gagal")

s, w, n, e = BBOX
types_re = "|".join(HIGHWAY_ORDINAL)
query = f'[out:json][timeout:180];(way["highway"~"^({types_re})$"]({s},{w},{n},{e}););out geom tags;'
elements = fetch_overpass(query)["elements"]

rows = [
    {"highway_type": el.get("tags", {}).get("highway"),
     "geometry": LineString([(p["lon"], p["lat"]) for p in el["geometry"]])}
    for el in elements
    if el.get("type") == "way" and len(el.get("geometry", [])) >= 2
]
roads = gpd.GeoDataFrame(rows, crs=CRS_WGS84)
roads["ordinal"] = roads["highway_type"].map(HIGHWAY_ORDINAL)
roads = roads.dropna(subset=["ordinal"]).to_crs(CRS_UTM49S).reset_index(drop=True)  # meter
print(f"Ruas jalan valid: {len(roads)}")

Ruas jalan valid: 79539


In [5]:
flood_pts = gpd.GeoDataFrame(
    floods, geometry=gpd.points_from_xy(floods["lon"], floods["lat"]), crs=CRS_WGS84
).to_crs(CRS_UTM49S)

def classify(pt):
    within = roads[roads.intersects(pt.buffer(SEARCH_RADIUS_M))]
    if len(within):                                     # ordinal tertinggi dalam radius
        return int(within.loc[within["ordinal"].idxmax(), "ordinal"])
    return int(roads.loc[roads.distance(pt).idxmin(), "ordinal"])  # fallback ruas terdekat

floods["road_class"] = [classify(g) for g in flood_pts.geometry]
print(floods["road_class"].value_counts().sort_index().to_string())

road_class
1     3
2    12
3    10
4     6
5     3


## 2. Jarak ke faskes terdekat via OSRM (C3 -> `dist_faskes_m`)

Pre-filter K faskes terdekat secara Haversine (hemat query), lalu ambil jarak
jaringan jalan OSRM terkecil di antara kandidat.

In [6]:
faskes = read_points(FASKES_FILE)
print(f"Faskes: {len(faskes)}")

def haversine(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    a = np.sin((lat2 - lat1) / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2) ** 2
    return EARTH_RADIUS_M * 2 * np.arcsin(np.sqrt(a))

# Matriks Haversine genangan x faskes, ambil indeks K terdekat per genangan.
hav = np.vstack([haversine(la, lo, faskes["lat"].values, faskes["lon"].values)
                 for la, lo in zip(floods["lat"], floods["lon"])])
cand = [np.argsort(row)[:K_CANDIDATES] for row in hav]

def osrm_route_m(lat1, lon1, lat2, lon2):
    # jarak jaringan jalan (meter); None kalau gagal
    url = f"{OSRM_BASE}/route/v1/driving/{lon1:.6f},{lat1:.6f};{lon2:.6f},{lat2:.6f}?overview=false"
    for _ in range(MAX_RETRIES):
        try:
            d = requests.get(url, timeout=30).json()
            if d.get("code") == "Ok" and d.get("routes"):
                return d["routes"][0]["distance"]
        except Exception:
            pass
        time.sleep(RETRY_WAIT_S)
    return None

dist_faskes = []
for i, (la, lo) in enumerate(tqdm(list(zip(floods["lat"], floods["lon"])), desc="OSRM faskes")):
    best = np.inf
    for j in cand[i]:
        d = osrm_route_m(la, lo, faskes["lat"].iloc[j], faskes["lon"].iloc[j])
        if d is not None:
            best = min(best, d)
        time.sleep(REQUEST_DELAY_S)
    best = hav[i, cand[i]].min() if best == np.inf else best   # fallback Haversine jika semua gagal
    dist_faskes.append(round(float(best), 2))

floods["dist_faskes_m"] = dist_faskes
print(floods["dist_faskes_m"].describe().round(1).to_string())

Faskes: 404


OSRM faskes: 100%|██████████| 34/34 [06:33<00:00, 11.58s/it]


count      34.0
mean      656.5
std       520.5
min         0.0
25%       262.9
50%       491.6
75%      1059.9
max      1926.8


## 3. Merge atribut -> `floods_final.csv`

Susun ke skema kolom yang dibaca `backend/app/main.py` `_load_floods`, lalu validasi.

In [7]:
# datetime = tanggal (date) + jam dari terima_berita; pakai kolom datetime bila sudah ada.
if "datetime" in floods.columns:
    dt = pd.to_datetime(floods["datetime"], errors="coerce")
else:
    tgl = pd.to_datetime(floods["tanggal"], errors="coerce")
    tb = pd.to_datetime(floods["terima_berita"], errors="coerce")
    dt = tgl.dt.normalize() + (tb - tb.dt.normalize()).fillna(pd.Timedelta(0))

# Susun ke skema yang dibaca backend main.py _load_floods
final = pd.DataFrame({
    "Datetime": dt.dt.strftime("%Y-%m-%d %H:%M:%S"),   # tanggal + waktu, mis. 2026-06-22 05:14:00
    "Latitude": floods["lat"],
    "Longitude": floods["lon"],
    "Deskripsi": floods.get("detail", floods.get("jalan", "")),
    "Images": "",                                       # data Damkar tidak menyertakan gambar
    "Ketinggian (cm)": floods["depth_cm"],
    "road_class": floods["road_class"].astype(int),
    "dist_faskes_m": floods["dist_faskes_m"].round(2),
})

for col in ["Latitude", "Longitude", "Ketinggian (cm)", "road_class", "dist_faskes_m"]:
    assert final[col].notna().all(), f"kolom {col} ada yang kosong"

final.to_csv(OUT / "floods_final.csv", index=False)
print(f"Tersimpan {OUT / 'floods_final.csv'}  shape={final.shape}")
final.head()

Tersimpan output_pipeline\floods_final.csv  shape=(34, 8)


,Datetime,Latitude,Longitude,Deskripsi,Images,Ketinggian (cm),road_class,dist_faskes_m
0,2026-06-22 05:14:00,-7.316784,112.729173,Genangan : 5 cm >> Surut>>,,5.0,2,171.3
1,2026-06-22 05:14:00,-7.262527,112.656512,Genangan : 10 cm = Surut Clear 0%,,10.0,2,465.0
2,2026-06-22 05:57:00,-7.297865,112.749232,Genangan : 20 cm,,20.0,4,1093.5
3,2026-06-22 06:31:00,-7.290469,112.745126,Genangan : 20 cm,,20.0,5,518.2
4,2026-06-22 06:43:00,-7.288979,112.760230,Genangan : 5 cm,,5.0,3,105.4


## 4. Matriks jarak & waktu OSRM

Urutan node **wajib** `[depo, genangan, IF]` (sama dengan `instance.py`). Query
OSRM Table per blok, sisa sel yang gagal diisi fallback Manhattan (seperti `geo.py`).

In [8]:
depots, ifs = read_points(DEPO_FILE), read_points(IF_FILE)
lats = np.concatenate([depots["lat"], floods["lat"], ifs["lat"]])
lons = np.concatenate([depots["lon"], floods["lon"], ifs["lon"]])
n_total = len(lats)
print(f"Node: {len(depots)} depo + {len(floods)} genangan + {len(ifs)} IF = {n_total}")

coord_str = ";".join(f"{lo:.6f},{la:.6f}" for la, lo in zip(lats, lons))  # OSRM: lon,lat

def osrm_table(src, dst):
    url = (f"{OSRM_BASE}/table/v1/driving/{coord_str}"
           f"?sources={';'.join(map(str, src))}&destinations={';'.join(map(str, dst))}"
           f"&annotations=distance,duration")
    for _ in range(MAX_RETRIES):
        try:
            d = requests.get(url, timeout=60).json()
            if d.get("code") == "Ok":
                return d["distances"], d["durations"]
        except Exception:
            pass
        time.sleep(RETRY_WAIT_S)
    return None, None

blocks = [list(range(i, min(i + BLOCK_SIZE, n_total))) for i in range(0, n_total, BLOCK_SIZE)]
dist = np.full((n_total, n_total), np.nan)
dur = np.full((n_total, n_total), np.nan)
for sb in tqdm(blocks, desc="OSRM table"):
    for db in blocks:
        ds, du = osrm_table(sb, db)
        if ds is not None:
            for si, s in enumerate(sb):
                for di, d in enumerate(db):
                    dist[s, d], dur[s, d] = ds[si][di], du[si][di]
        time.sleep(REQUEST_DELAY_S)
print(f"Sel kosong (perlu fallback): {int(np.isnan(dist).sum())}")

Node: 12 depo + 34 genangan + 140 IF = 186


OSRM table: 100%|██████████| 5/5 [01:00<00:00, 12.14s/it]

Sel kosong (perlu fallback): 0


In [9]:
# Fallback Manhattan (asumsi 30 km/jam, konsisten dgn geo.py) untuk sel yg gagal OSRM.
lat_r, lon_r = np.deg2rad(lats), np.deg2rad(lons)
cos_avg = np.cos(lat_r.mean())
man = EARTH_RADIUS_M * (np.abs(lat_r[:, None] - lat_r[None, :])
                        + np.abs(lon_r[:, None] - lon_r[None, :]) * cos_avg)
dist[np.isnan(dist)] = man[np.isnan(dist)]
dur[np.isnan(dur)] = (man / (30_000 / 3600))[np.isnan(dur)]
np.fill_diagonal(dist, 0.0); np.fill_diagonal(dur, 0.0)

np.save(OUT / "distance_matrix.npy", dist)
np.save(OUT / "time_matrix.npy", dur)
print(f"Tersimpan matriks {dist.shape} -> {OUT}/distance_matrix.npy, {OUT}/time_matrix.npy")

Tersimpan matriks (186, 186) -> output_pipeline/distance_matrix.npy, output_pipeline/time_matrix.npy


## Deploy

1. `output_pipeline/floods_final.csv` -> `backend/app/data/floods.csv`
2. `output_pipeline/distance_matrix.npy` & `time_matrix.npy` -> `backend/app/data/`
3. Restart backend (`main.py` memuat file ini otomatis di `lifespan()`).

Jalankan ulang seluruh notebook hanya bila dataset genangan/depo/IF berubah,
karena ukuran & urutan matriks bergantung pada `n_total`.